In [ ]:
import cv2
import numpy as np
import tifffile as tiff
import matplotlib.pyplot as plt

# 1. Читаем снимок
img = tiff.imread("/content/rec_00440.tif").astype(np.float32)

# 2. Нормализация в 8-бит uint8 (0-255)
img_norm = (img - np.min(img)) / (np.max(img) - np.min(img) + 1e-8)
img_uint8 = (img_norm * 255).astype(np.uint8)

# 3. Базовый контраст и фильтрация (как в первом удачном коде)
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
img_clahe = clahe.apply(img_uint8)
denoised = cv2.bilateralFilter(img_clahe, d=9, sigmaColor=75, sigmaSpace=75)

# 4. Canny + Склейка
edges_raw = cv2.Canny(denoised, threshold1=65, threshold2=140)
kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
edges_connected = cv2.morphologyEx(edges_raw, cv2.MORPH_CLOSE, kernel_close)

# 5. Маскирование внешнего контура керна
_, body_mask = cv2.threshold(denoised, 20, 255, cv2.THRESH_BINARY)
kernel_erode = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (25, 25))
inner_body_mask = cv2.erode(body_mask, kernel_erode)
internal_edges = cv2.bitwise_and(edges_connected, edges_connected, mask=inner_body_mask)

# ==============================================================================
# 🔥 ГЛАВНОЕ: Оставляем ТОЛЬКО вытянутые трещины (Вся синяя рябь удаляется!)
# ==============================================================================
num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(internal_edges, connectivity=8)

# Создаем чистую Ч/Б маску
clean_cracks_only_mask = np.zeros_like(internal_edges)

for i in range(1, num_labels):
    area = stats[i, cv2.CC_STAT_AREA]
    w = stats[i, cv2.CC_STAT_WIDTH]
    h = stats[i, cv2.CC_STAT_HEIGHT]
    aspect_ratio = max(w, h) / (min(w, h) + 1e-5)

    # 1. ТРЕЩИНЫ: только вытянутые связные линии (aspect_ratio >= 2.2 и длина >= 25px)
    # Вся круглая мелкая рябь цемента здесь АВТОМАТИЧЕСКИ ИГНОРИРУЕТСЯ:
    if aspect_ratio >= 2.2 and area >= 25:
        clean_cracks_only_mask[labels == i] = 255
    # 2. Крупные контуры (например, контуры больших дефектов):
    elif area >= 350:
        clean_cracks_only_mask[labels == i] = 255

# ==============================================================================
# ОТРИСОВКА: Исходник слева vs Чистая Ч/Б маска трещин справа
# ==============================================================================
fig, axes = plt.subplots(1, 2, figsize=(20, 10))

axes[0].imshow(img_uint8, cmap='gray')
axes[0].set_title("1. Исходный эталонный срез КТ", fontsize=14)
axes[0].axis('off')

axes[1].imshow(clean_cracks_only_mask, cmap='gray')
axes[1].set_title("2. Итоговая Ч/Б маска микротрещин (Без шума и пор)", fontsize=14, color='darkgreen')
axes[1].axis('off')

plt.tight_layout()
plt.show()

# Сохраняем готовую маску
cv2.imwrite("final_clean_cracks_mask.png", clean_cracks_only_mask)
print("🎉 Готовая Ч/Б маска без лишнего шума сохранена в 'final_clean_cracks_mask.png'!")